# Transformer NQS sweep — 2D toric code (L=4, OBC)

Train and compare neural-quantum-state architectures on the **2D toric code** at
**L=4, OBC (N=24 qubits)**, field **hx = hz = 0.2**, and log everything to
**Weights & Biases** so runs can be compared against **exact diagonalization**.

At N=24 the Hilbert space is 2^24 = 16.7M states, small enough for a **Krylov
(Lanczos) ED** ground-state energy on Colab RAM — that gives the exact reference
`E_exact`, and every run logs the error `delta = |E - E_exact| / |E_exact|`.

**Architectures**
- **Combo** — the Approximately-Symmetric NQS (A_v baked in via the Wilson product); the baseline.
- **Factored-attention transformer** — a 4-layer ViT with a learnable per-head token-mixing matrix (no Q/K); tunable heads / layers / residual-stream dim / lr / diag_shift / n_iter.

**Before you run**
1. Push the branch that contains `model/transformer.py`, `model/builders_2d.py`, `model/train_2d.py`, then set `REPO_URL` and `BRANCH` in the setup cell (use a Personal Access Token in the URL for a private repo).
2. **Runtime → Change runtime type → GPU** (T4/A100). VMC + minSR is much faster on GPU.


## 1. Setup — install, clone, enable float64, W&B login

In [ ]:
!pip -q install netket wandb optax

import os, sys

# --- set me -----------------------------------------------------------------
REPO_URL = "https://github.com/<YOUR_USER>/Approximate-Symmetries-TC.git"  # PAT-in-URL for private
BRANCH   = "nersc-gridinv-runs"     # the branch that has model/transformer.py etc.
# ---------------------------------------------------------------------------
REPO_DIR = "/content/Approximate-Symmetries-TC"

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

import jax
jax.config.update("jax_enable_x64", True)          # float64 SR/QGT
print("jax", jax.__version__, "| devices:", jax.devices())

import wandb
wandb.login()                                       # paste your API key when prompted

## 2. Geometry, Hamiltonian, and settings

In [ ]:
import numpy as np
import netket as nk

from model.geometry import ToricCodeGeometry
from model.hamiltonian import create_hamiltonian

# --- system + field ---------------------------------------------------------
L, BC, HX, HZ, J = 4, "OBC", 0.2, 0.2, 1.0
# ---------------------------------------------------------------------------

geo = ToricCodeGeometry(L, L, BC)
hi  = nk.hilbert.Spin(s=1/2, N=geo.N)
Ham = create_hamiltonian(hi, geo.vertex_all, geo.plaq_all, geo.bonds,
                         hx=HX, hy=0.0, hz=HZ, J=J, dtype="float64")

print(f"N = {geo.N} qubits  (2^N = {2**geo.N:,} states)")

## 3. Exact diagonalization (Krylov / Lanczos)

Matrix-free Hamiltonian (`model/exact_diag.py`, Numba kernels) fed to
`scipy.sparse.linalg.eigsh`. At N=24 this uses ~2–3 GB and a few minutes — run
it **once**; `E_exact` is reused by every training run below.

In [ ]:
from model.exact_diag import hamiltonian_linop
from scipy.sparse.linalg import eigsh

Hlin, _basis = hamiltonian_linop(geo, hx=HX, hy=0.0, hz=HZ, J=J)   # LinearOperator (2^N, 2^N)
E_exact = float(eigsh(Hlin, k=1, which="SA", return_eigenvectors=False)[0])
print("E_exact =", E_exact)

## 4. Experiment runner

Every run shares the **same sampler and optimization loop** (`build_sampler` +
`run_loop` from `model/builders_2d.py`) so only the *architecture* differs — a
clean apples-to-apples comparison. Per-step metrics (energy, `delta`, V-score,
R_hat, acceptance) stream to W&B; final observables go to the run summary.

`WANDB_GROUP` ties the runs together so they overlay on one W&B plot.

In [ ]:
import time, hashlib
from model.builders_2d import build_sampler, run_loop
from model.train_2d import nqs_observables

# --- W&B target -------------------------------------------------------------
WANDB_PROJECT = "approx-sym-2D-TC-transformer"
WANDB_ENTITY  = None                      # your entity/team, or None for default
WANDB_GROUP   = f"L{L}_{BC}_hx{HX}_hz{HZ}"
N_CHAINS      = 16                        # bump to 256+ on GPU for lower-variance grads
# ---------------------------------------------------------------------------

sampler = build_sampler({"n_chains": N_CHAINS}, hi, geo)   # shared across all runs

def run_experiment(name, model, *, n_iter, dt, diag_shift, qgt,
                   lr_min=None, n_samples=4096, n_discard=8, chunk_size=None,
                   tags=None, extra_cfg=None):
    vs = nk.vqs.MCState(sampler, model, n_samples=n_samples,
                        n_discard_per_chain=n_discard, chunk_size=chunk_size)
    cfg = dict(L=L, bc=BC, hx=HX, hz=HZ, arch=name, n_iter=n_iter, dt=dt,
               lr_min=lr_min, diag_shift=diag_shift, qgt=qgt, n_samples=n_samples,
               n_chains=N_CHAINS, n_params=int(vs.n_parameters), E_exact=E_exact,
               **(extra_cfg or {}))
    print(f"[{name}] n_params = {cfg['n_params']:,}  qgt = {qgt}")

    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, group=WANDB_GROUP,
                     name=name, tags=tags, config=cfg, reinit=True,
                     id=hashlib.md5(name.encode()).hexdigest()[:12], resume="allow")

    def on_step(step, E, vs):
        e   = float(np.real(E.mean)); de = float(np.real(E.error_of_mean))
        var = float(np.real(E.variance))
        acc = float(vs.sampler_state.n_accepted) / max(1, float(vs.sampler_state.n_steps))
        wandb.log({"step": step, "energy": e, "energy_error": de,
                   "energy_variance": var,
                   "energy_abs_err": abs(e - E_exact),
                   "delta": abs(e - E_exact) / abs(E_exact),
                   "Vscore": geo.N * var / e**2,
                   "R_hat": float(np.real(E.R_hat)),
                   "tau_corr": float(np.real(E.tau_corr)),
                   "mcmc_acceptance": acc}, step=step)

    t0 = time.time()
    run_loop(vs, Ham, n_iter=n_iter, dt=dt, diag_shift=diag_shift,
             lr_min=lr_min, qgt=qgt, on_step=on_step)
    obs = nqs_observables(vs, Ham, geo)
    obs["E_exact"] = E_exact
    obs["delta"]   = abs(obs["E0"] - E_exact) / abs(E_exact)
    obs["runtime_s"] = time.time() - t0
    for k, v in obs.items():
        run.summary[k] = v
    run.finish()
    print(f"[{name}] E = {obs['E0']:.4f}  delta = {obs['delta']:.3e}  "
          f"Vscore = {obs['Vscore']:.2e}  ({obs['runtime_s']:.0f}s)")
    return obs

## 5. Combo baseline (Approximately-Symmetric NQS)

The published architecture: a non-symmetric edge-CNN → **Wilson 4-product**
(bakes in A_v) → invariant plaquette-CNN. Tiny (~hundreds of params); `dense`
QGT is the right SR here (n_params << n_samples).

In [ ]:
from model.networks import create_model, KernelManager

combo_cfg = dict(architecture="Combo",
                 channels_noninv=[1, 16], channels_inv=[16, 8, 1],
                 kernel_size=2, bc=BC, dtype="float64", rescale=1.0)

km = KernelManager(L, L, BC, combo_cfg["kernel_size"], L - 1,
                   geo.arr_coord, geo.dg_p, geo.N)
combo_model = create_model(combo_cfg, geo.plaq_all, km)

run_experiment("Combo", combo_model,
               n_iter=400, dt=2e-2, lr_min=2e-3, diag_shift=2e-4, qgt="dense",
               tags=["Combo"], extra_cfg={k: str(v) for k, v in combo_cfg.items()})

## 6. Transformer — single run (tune the knobs here)

Architecture knobs: `d` (residual-stream / embedding dim), `n_heads`, `n_layers`,
`mlp_ratio` (MLP hidden = mlp_ratio·d). Optimization knobs: `dt` (initial lr),
`lr_min` (cosine floor), `diag_shift` (SR regularizer), `n_iter`, `n_samples`.
`qgt="minsr"` (the sample-space S-matrix trick) is correct because the
transformer's n_params exceeds n_samples. **`d` must be divisible by `n_heads`.**

In [ ]:
from model.transformer import FactoredAttentionWavefunction

arch = dict(d=32, n_heads=4, n_layers=4, mlp_ratio=2)      # <-- residual stream, heads, layers
model = FactoredAttentionWavefunction(N=geo.N, **arch)

run_experiment(f"factored_d{arch['d']}_h{arch['n_heads']}_l{arch['n_layers']}",
               model,
               n_iter=500, dt=2e-2, lr_min=2e-3, diag_shift=1e-3, qgt="minsr",
               n_samples=4096, tags=["factored_transformer"], extra_cfg=arch)

## 7. Hyperparameter sweep

Each entry is one W&B run in the same group, so they overlay for comparison.
Edit the list freely — architecture dims and optimization settings per run.

In [ ]:
sweep = [
    dict(d=16, n_heads=4, n_layers=4, dt=2e-2, diag_shift=1e-3),
    dict(d=32, n_heads=4, n_layers=4, dt=2e-2, diag_shift=1e-3),
    dict(d=32, n_heads=8, n_layers=6, dt=1e-2, diag_shift=1e-3),
    dict(d=48, n_heads=6, n_layers=6, dt=1e-2, diag_shift=3e-3),
]

for s in sweep:
    arch = dict(d=s["d"], n_heads=s["n_heads"], n_layers=s["n_layers"], mlp_ratio=2)
    m = FactoredAttentionWavefunction(N=geo.N, **arch)
    name = f"factored_d{arch['d']}_h{arch['n_heads']}_l{arch['n_layers']}_ds{s['diag_shift']}"
    run_experiment(name, m,
                   n_iter=500, dt=s["dt"], lr_min=s["dt"] / 10.0,
                   diag_shift=s["diag_shift"], qgt="minsr", n_samples=4096,
                   tags=["factored_transformer", "sweep"], extra_cfg={**arch, **s})

## Notes

- **Compare in W&B:** open the `L4_OBC_hx0.2_hz0.2` group; plot `delta` (or
  `energy_abs_err`) vs `step` to rank architectures against ED, and check the run
  summary `n_params` to weigh accuracy against parameter cost.
- **Expected Step-1 finding:** the transformer (no symmetry) should approach
  `E_exact` but needs far more parameters than Combo to match its accuracy — that
  gap is what circulant/PBC translation invariance (Step 2) and the Wilson
  sandwich (Step 3) are meant to close.
- **If you hit OOM** at large `d`/`n_samples`, set `chunk_size=2048` in
  `run_experiment`, or lower `n_samples`.
- **Krylov ED** is the only exact anchor here; it works at L=4 (2^24) but not
  larger — beyond that, rank runs by V-score.